## Centerline

In [37]:
import numpy
import sys
from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
import vtk
from sklearn.decomposition import PCA
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import BSpline
from scipy import ndimage
import trimesh
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline, getSurfaceMesh, tetra_mesh_from_stl, set_the_offset

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/"

eyeball_mesh = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/Segmentation_1_surf_id_1_inner.stl"
lense_mesh = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/re-meshed_h_1.0/id_7__h_1.0.stl"

offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")
pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

out_folder = "extended_muscles/"
centerline_folder = "centerlines/"

mask_filename_base = "Segmentation_1_filtered_surf_id_" # + _1.npy

muslce_ids = [2,3,4,5]

eyeball_id = 1
lense_id = 7

mask_eyeball = numpy.load(base + mask_filename_base + str(eyeball_id) + ".npy")



## 1. Lense centre & axis

In [38]:
mesh = trimesh.load(lense_mesh)
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


projections_0 = (mesh.vertices - lense_center) @ vh[0]
lense_diameter = (projections_0.max() - projections_0.min())*pixel_spacing[0]

projections_1 = (mesh.vertices - lense_center) @ vh[-1]
lense_thickness = (projections_1.max() - projections_1.min())*pixel_spacing[0]



print("lense_diameter = ", lense_diameter)


def eyeball_axis(t):
    return lense_center + t * axis_direction


axis_points = [eyeball_axis(-20.0), eyeball_axis(20.0)]

export_centerline(axis_points, 1.0 , base + out_folder + "axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

lense_center =  [ 1.01680029  8.08340236 -2.5252567 ]
lense_diameter =  3.9438819228328748


## 2. Fit two planes

In [39]:
def load_centerline(base, centerline, cid):
    data = numpy.load(base + centerline + "centerline_id_"+str(cid)+"_parametric.npz")

    
    sx = BSpline(data["tx"], data["cx"], data["kx"])
    sy = BSpline(data["ty"], data["cy"], data["ky"])
    sz = BSpline(data["tz"], data["cz"], data["kz"])
    
    return sx, sy, sz

centerlines = [load_centerline(base, centerline_folder, cid) for cid in muslce_ids]

In [40]:
def sample_spline(sx, sy, sz, n=100):
    t_min = max(sx.t[0], sy.t[0], sz.t[0])
    t_max = min(sx.t[-1], sy.t[-1], sz.t[-1])
    
    t = numpy.linspace(t_min, t_max, n)
    
    x = sx(t)
    y = sy(t)
    z = sz(t)
    
    return numpy.vstack((x, y, z)).T  # shape (n, 3)

def fit_plane(points):
    # points: (N, 3)
    centroid = points.mean(axis=0)
    
    # subtract centroid
    X = points - centroid
    
    # SVD
    _, _, vh = numpy.linalg.svd(X)
    
    normal = vh[-1]  # smallest singular vector
    
    return centroid, normal


pairs = [(0, 2), (1, 3)]

planes = []

for i, j in pairs:
    pts1 = sample_spline(*centerlines[i])
    pts2 = sample_spline(*centerlines[j])
    
    all_pts = numpy.vstack((pts1, pts2, lense_center[None, :]))
    
    point, normal = fit_plane(all_pts)
    
    planes.append((point, normal))


print(numpy.dot(planes[0][1], planes[1][1] ))

0.06065041362809068


In [41]:
n3 = numpy.cross(planes[0][1], planes[1][1])
n3 = n3 / numpy.linalg.norm(n3)



## 3. Eyeball 

In [42]:
mesh_eye = trimesh.load(eyeball_mesh)

mesh_eye.vertices = mesh_eye.vertices - offset

side = (mesh_eye.vertices - lense_center) @ axis_direction
eps = 1e-6
labels = numpy.zeros(len(mesh_eye.vertices))
labels[side > eps] = 1
labels[side < -eps] = 0

print(labels.shape)

meshio.write(
    base + out_folder + "eyeball_with_labels.vtk",
    meshio.Mesh(
        points=mesh_eye.vertices,
        cells=[("triangle", mesh_eye.faces)],
        point_data={"side": labels}
    )
)

(12080,)


In [43]:
section = mesh_eye.section(
    plane_origin=lense_center - lense_thickness*axis_direction,
    plane_normal=axis_direction
)


# path_2D, to_3D = section.to_planar()

# pts_2d = path_2D.vertices

# # convert (N,2) → (N,3) by adding zero coordinate in plane frame
# pts_3d_local = numpy.column_stack([pts_2d, numpy.zeros(len(pts_2d))])

# # apply transform
# curve_3D = trimesh.transform_points(pts_3d_local, to_3D)

# export

section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + out_folder + "section_curve.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)


### Extention

In [44]:
def closest_point_on_curve(p_end, curve):
    diff = curve - p_end
    dist2 = numpy.sum(diff**2, axis=1)
    idx = numpy.argmin(dist2)
    return curve[idx], idx

def find_intersection_with_mask(start, direction, mask, step=0.5, max_dist=100):
    p = start.copy()
    
    for _ in range(int(max_dist / step)):
        p += direction * step
        idx = numpy.round(p).astype(int)
        
        # check bounds
        if (
            0 <= idx[0] < mask.shape[0] and
            0 <= idx[1] < mask.shape[1] and
            0 <= idx[2] < mask.shape[2]
        ):
            if mask[tuple(idx)] == 1:
                return p  # first contact
        else:
            break
    
    return None

i = 3

pts = sample_spline(*centerlines[i])
p_end = pts[-1]*pixel_spacing - offset # last point of spline (3D)

print("p_end = ", p_end)

target, idx = closest_point_on_curve(p_end, section.vertices)

print("target = ", target)




# direction = target - p_end
# direction = direction / numpy.linalg.norm(direction)
# D = find_intersection_with_mask((p_end+offset)/pixel_spacing[0], direction, mask_eyeball)

# print("D = ", D)

# def generate_extension(p0, p1, n_points=50):
#     t = numpy.linspace(0, 1, n_points)
#     return p0[None, :] + (p1 - p0)[None, :] * t[:, None]

# extension = generate_extension((p_end+offset)/pixel_spacing[0], D)


p_end =  [ -5.21584594  -9.25462159 -14.66157888]
target =  [ -3.73139681   3.84925558 -12.92379469]


In [45]:
def bezier_quadratic(P0, P1, P2, n=50):
    t = numpy.linspace(0, 1, n)
    return (
        ((1 - t)**2)[:, None] * P0 +
        (2 * (1 - t) * t)[:, None] * P1 +
        (t**2)[:, None] * P2
    )

tangent = pts[-1] - pts[-2]
tangent = tangent / numpy.linalg.norm(tangent)

alpha = 8  # controls curvature strength

P0 = p_end
P2 = target
P1 = P0 + alpha * tangent

extension = bezier_quadratic(P0, P1, P2)

## Option 1: Extended muscle shape

In [ ]:
# --------------------------------------------------
# 1. Build local frame from tangent
# --------------------------------------------------
def build_frame(tangent):
    tangent = tangent / numpy.linalg.norm(tangent)
    
    tmp = numpy.array([1.0, 0.0, 0.0])
    if abs(numpy.dot(tmp, tangent)) > 0.9:
        tmp = numpy.array([0.0, 1.0, 0.0])
    
    u = numpy.cross(tangent, tmp)
    u /= numpy.linalg.norm(u)
    
    v = numpy.cross(tangent, u)
    
    return u, v, tangent


# --------------------------------------------------
# 2. Extract cross-section from existing muscle
# --------------------------------------------------
def extract_cross_section(mask, pts, offset, pixel_spacing, radius=6):
    
    # take point slightly before the end
    p_ref = pts[-10]
    p_ref_vox = numpy.round(p_ref).astype(int)
    
    # tangent
    tangent = pts[-1] - pts[-2]
    u, v, _ = build_frame(tangent)
    
    cross_section = []
    
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            for dz in range(-radius, radius+1):
                
                p = p_ref_vox + numpy.array([dx, dy, dz])
                
                if (
                    0 <= p[0] < mask.shape[0] and
                    0 <= p[1] < mask.shape[1] and
                    0 <= p[2] < mask.shape[2]
                ):
                    if mask[tuple(p)] == 1:
                        
                        rel = p - p_ref_vox
                        
                        x = numpy.dot(rel, u)
                        y = numpy.dot(rel, v)
                        
                        cross_section.append([x, y])
    
    return numpy.array(cross_section)


# --------------------------------------------------
# 3. Sweep cross-section along extension curve
# --------------------------------------------------
def sweep_extension(extension, cross_section, new_mask, offset, pixel_spacing):
    
    for i in range(len(extension)):
        
        p = extension[i]
        p_vox = (p + offset) / pixel_spacing
        
        # compute tangent along curve
        if i == 0:
            tangent = extension[i+1] - extension[i]
        elif i == len(extension) - 1:
            tangent = extension[i] - extension[i-1]
        else:
            tangent = extension[i+1] - extension[i-1]
        
        u, v, _ = build_frame(tangent)
        
        for (x, y) in cross_section:
            
            point = p_vox + x * u + y * v
            idx = numpy.round(point).astype(int)
            
            if (
                0 <= idx[0] < new_mask.shape[0] and
                0 <= idx[1] < new_mask.shape[1] and
                0 <= idx[2] < new_mask.shape[2]
            ):
                new_mask[tuple(idx)] = 1


# --------------------------------------------------
# 4. MAIN PIPELINE
# --------------------------------------------------

# new mask
mask_muscle = numpy.load(base + mask_filename_base + str(muslce_ids[i]) + ".npy")

new_mask = numpy.zeros_like(mask_eyeball)


# extract cross-section from original muscle
cross_section = extract_cross_section(
    mask=mask_muscle,        # original muscle mask
    pts=pts,                 # spline points
    offset=offset,
    pixel_spacing=pixel_spacing,
    radius=6
)

print("Cross-section points:", len(cross_section))

# sweep along extension curve
sweep_extension(
    extension=extension,     # Bézier curve (world coords)
    cross_section=cross_section,
    new_mask=new_mask,
    offset=offset,
    pixel_spacing=pixel_spacing
)

numpy.save(base + out_folder +"mask_extention_"+ str(muslce_ids[i]), new_mask)


Cross-section points: 1177


In [48]:
h=1

new_mask = ndimage.median_filter(new_mask, size=3) 


mesh_surf = getSurfaceMesh(new_mask, base + out_folder +"_extention_surf_id_"+ str(muslce_ids[i]) + ".stl", pixel_spacing[0], False )

tetra_mesh_from_stl(base + out_folder + "_extention_surf_id_"+ str(muslce_ids[i]) +".stl", base + out_folder + "_extention_surf_id_" + str(muslce_ids[i]) + "_h_"+str(h), h)
set_the_offset(base  + out_folder + "_extention_surf_id_" + str(muslce_ids[i]) + "_h_"+str(h), offset)

Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/extended_muscles/_extention_surf_id_5.stl
Info    : Reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/extended_muscles/_extention_surf_id_5.stl'...
Info    : 4456 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/extended_muscles/_extention_surf_id_5.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 4456 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.0392202s, CPU 0.038355s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of disc

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.